# Football Team Performance Profiling — Team-Season Feature Engineering

## Objective

This notebook converts the existing match-level engineered dataset into one observation per team and season. The resulting features describe squad quality, tactics, recent form, and historical season performance for later exploratory analysis and unsupervised profiling.

A sports organisation can use these team-season records to compare similar teams, identify shared strengths and weaknesses, and support benchmarking. The source match dataset remains unchanged.

## 1. Imports and Project Paths

All paths are resolved from the repository root so the notebook works when launched from either the project root or a notebook directory.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
INPUT_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "model_data_engineered.csv"
TEAM_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "team_season_features.csv"

assert INPUT_DATA_PATH.is_file(), f"Input dataset not found: {INPUT_DATA_PATH}"

## 2. Load and Validate the Engineered Match Data

The source must contain one unique row per match and complete team, season, league, score, and profiling fields. These checks prevent incomplete match records from being silently carried into the team-season aggregation.

In [2]:
match_data = pd.read_csv(INPUT_DATA_PATH)

profile_features = [
    "buildUpPlaySpeed", "buildUpPlayPassing", "chanceCreationPassing",
    "chanceCreationCrossing", "chanceCreationShooting", "defencePressure",
    "defenceAggression", "defenceTeamWidth", "form_points_5",
    "avg_goals_scored_5", "avg_goals_conceded_5", "players_with_attributes",
    "avg_overall_rating", "avg_potential", "goalkeeper_rating", "avg_attack",
    "avg_passing", "avg_technical", "avg_pace", "avg_physical", "avg_defence",
]
key_columns = [
    "id", "date", "season", "stage", "league_id", "league_name",
    "home_team_api_id", "away_team_api_id", "home_team_name", "away_team_name",
    "home_team_goal", "away_team_goal",
]
required_columns = key_columns + [
    f"{side}_{feature}"
    for side in ("home", "away")
    for feature in profile_features
]

missing_columns = sorted(set(required_columns) - set(match_data.columns))
assert not missing_columns, f"Missing required columns: {missing_columns}"

source_checks = pd.Series({
    "matches": len(match_data),
    "seasons": match_data["season"].nunique(),
    "leagues": match_data["league_name"].nunique(),
    "unique_teams": pd.unique(pd.concat([
        match_data["home_team_api_id"], match_data["away_team_api_id"]
    ], ignore_index=True)).size,
    "duplicate_match_ids": match_data["id"].duplicated().sum(),
    "missing_required_values": int(match_data[required_columns].isna().sum().sum()),
}, name="value")
display(source_checks.to_frame())
display(match_data[[
    "season", "league_name", "home_team_name", "away_team_name",
    "home_team_goal", "away_team_goal", "home_avg_overall_rating",
    "away_avg_overall_rating",
]].head())

assert len(match_data) == 17_609
assert source_checks["duplicate_match_ids"] == 0
assert source_checks["missing_required_values"] == 0

,value
matches,17609
seasons,6
leagues,11
unique_teams,262
duplicate_match_ids,0
missing_required_values,0


,season,league_name,home_team_name,away_team_name,home_team_goal,away_team_goal,home_avg_overall_rating,away_avg_overall_rating
0,2010/2011,Switzerland Super League,Grasshopper Club Zürich,Neuchâtel Xamax,1,1,57.545455,60.545455
1,2010/2011,Switzerland Super League,AC Bellinzona,FC Sion,0,2,59.363636,62.363636
2,2010/2011,Switzerland Super League,FC Luzern,FC St. Gallen,4,0,60.636364,58.545455
3,2010/2011,Switzerland Super League,FC Basel,FC Zürich,3,2,66.818182,63.363636
4,2010/2011,Switzerland Super League,FC Sion,FC Basel,1,2,62.000000,66.727273


### Finding

The input contains 17,609 valid matches with no duplicate match IDs or missing values in the fields needed for profiling. It is therefore suitable for reshaping without additional imputation.

## 3. Reshape Matches into Team-Match Observations

Each match contributes two observations: one from the home team's perspective and one from the away team's perspective. The same transformation is applied to both sides so goals, points, results, and profiling features have a common schema.

In [3]:
common_columns = ["id", "date", "season", "stage", "league_id", "league_name"]


def team_view(data, side):
    opponent = "away" if side == "home" else "home"
    selected = (
        common_columns
        + [f"{side}_team_api_id", f"{side}_team_name",
           f"{side}_team_goal", f"{opponent}_team_goal"]
        + [f"{side}_{feature}" for feature in profile_features]
    )
    renamed = {
        f"{side}_team_api_id": "team_api_id",
        f"{side}_team_name": "team_name",
        f"{side}_team_goal": "goals_for",
        f"{opponent}_team_goal": "goals_against",
        **{f"{side}_{feature}": feature for feature in profile_features},
    }
    team_rows = data[selected].rename(columns=renamed).copy()
    team_rows["venue"] = side.title()
    return team_rows


team_matches = pd.concat(
    [team_view(match_data, "home"), team_view(match_data, "away")],
    ignore_index=True,
)
team_matches["goal_difference"] = (
    team_matches["goals_for"] - team_matches["goals_against"]
)
team_matches["points"] = np.select(
    [
        team_matches["goals_for"] > team_matches["goals_against"],
        team_matches["goals_for"] == team_matches["goals_against"],
    ],
    [3, 1],
    default=0,
)
team_matches["result"] = np.select(
    [
        team_matches["goals_for"] > team_matches["goals_against"],
        team_matches["goals_for"] == team_matches["goals_against"],
    ],
    ["Win", "Draw"],
    default="Loss",
)
for result_name in ("win", "draw", "loss"):
    team_matches[result_name] = team_matches["result"].eq(result_name.title()).astype(int)

reshape_summary = pd.Series({
    "source_matches": len(match_data),
    "team_match_observations": len(team_matches),
    "unique_teams": team_matches["team_api_id"].nunique(),
    "team_season_combinations": team_matches[["team_api_id", "season"]].drop_duplicates().shape[0],
}, name="value")
display(reshape_summary.to_frame())
display(team_matches.head())

assert len(team_matches) == 35_218

,value
source_matches,17609
team_match_observations,35218
unique_teams,262
team_season_combinations,1076


,id,date,season,stage,league_id,league_name,team_api_id,team_name,goals_for,goals_against,buildUpPlaySpeed,buildUpPlayPassing,chanceCreationPassing,chanceCreationCrossing,chanceCreationShooting,defencePressure,defenceAggression,defenceTeamWidth,form_points_5,avg_goals_scored_5,avg_goals_conceded_5,players_with_attributes,avg_overall_rating,avg_potential,goalkeeper_rating,avg_attack,avg_passing,avg_technical,avg_pace,avg_physical,avg_defence,venue,goal_difference,points,result,win,draw,loss
0,24918,2010-07-17,2010/2011,1,24558,Switzerland Super League,9956,Grasshopper Club Zürich,1,1,30.0,50.0,55.0,70.0,55.0,40.0,60.0,60.0,9.0,2.4,0.6,11,57.545455,66.454545,56.6,47.425,52.800000,54.90,62.600000,56.800,49.100,Home,0,1,Draw,0,1,0
1,24920,2010-07-18,2010/2011,1,24558,Switzerland Super League,6493,AC Bellinzona,0,2,40.0,50.0,55.0,65.0,55.0,45.0,50.0,55.0,7.0,1.0,1.6,11,59.363636,63.909091,56.2,48.450,54.333333,53.75,59.200000,59.050,50.850,Home,-2,0,Loss,0,0,1
2,24921,2010-07-18,2010/2011,1,24558,Switzerland Super League,10199,FC Luzern,4,0,45.0,60.0,45.0,70.0,60.0,40.0,45.0,50.0,13.0,2.8,0.4,11,60.636364,69.818182,63.8,56.575,56.633333,58.00,65.933333,61.975,46.550,Home,4,3,Win,1,0,0
3,24922,2010-07-20,2010/2011,1,24558,Switzerland Super League,9931,FC Basel,3,2,40.0,45.0,50.0,65.0,50.0,60.0,60.0,60.0,12.0,2.4,0.8,11,66.818182,76.818182,64.0,56.500,63.300000,64.30,70.233333,66.250,59.300,Home,1,3,Win,1,0,0
4,24973,2010-07-24,2010/2011,2,24558,Switzerland Super League,10179,FC Sion,1,2,35.0,60.0,50.0,60.0,60.0,35.0,40.0,45.0,9.0,1.8,1.4,11,62.000000,70.181818,65.4,53.175,57.966667,55.80,62.600000,61.775,51.425,Home,-1,0,Loss,0,0,1


### Finding

The 17,609 matches become 35,218 team-match observations, exactly two per match. Goals and results are now expressed consistently from the focal team's perspective.

## 4. Aggregate to Team-Season Features

Squad, tactical, and recent-form characteristics are averaged across each season. Match totals are retained to calculate seven comparable per-match performance measures. These performance variables are intended mainly for post-cluster interpretation.

In [4]:
team_season = (
    team_matches
    .groupby(
        ["season", "league_id", "league_name", "team_api_id", "team_name"],
        as_index=False,
    )
    .agg(
        matches_played=("id", "count"),
        wins=("win", "sum"),
        draws=("draw", "sum"),
        losses=("loss", "sum"),
        total_points=("points", "sum"),
        goals_for=("goals_for", "sum"),
        goals_against=("goals_against", "sum"),
        goal_difference=("goal_difference", "sum"),
        **{feature: (feature, "mean") for feature in profile_features},
    )
)

team_season["points_per_match"] = team_season["total_points"] / team_season["matches_played"]
team_season["goals_for_per_match"] = team_season["goals_for"] / team_season["matches_played"]
team_season["goals_against_per_match"] = team_season["goals_against"] / team_season["matches_played"]
team_season["goal_difference_per_match"] = team_season["goal_difference"] / team_season["matches_played"]
team_season["win_rate"] = team_season["wins"] / team_season["matches_played"]
team_season["draw_rate"] = team_season["draws"] / team_season["matches_played"]
team_season["loss_rate"] = team_season["losses"] / team_season["matches_played"]

performance_variables = [
    "points_per_match", "goals_for_per_match", "goals_against_per_match",
    "goal_difference_per_match", "win_rate", "draw_rate", "loss_rate",
]
display(team_season[[
    "season", "league_name", "team_name", "matches_played", *performance_variables
]].head(10).round(3))

,season,league_name,team_name,matches_played,points_per_match,goals_for_per_match,goals_against_per_match,goal_difference_per_match,win_rate,draw_rate,loss_rate
0,2010/2011,Belgium Jupiler League,KAS Eupen,3,0.667,0.667,1.333,-0.667,0.000,0.667,0.333
1,2010/2011,Belgium Jupiler League,KV Mechelen,27,1.556,1.148,1.037,0.111,0.407,0.333,0.259
2,2010/2011,Belgium Jupiler League,Club Brugge KV,27,1.630,1.852,1.259,0.593,0.481,0.185,0.333
3,2010/2011,Belgium Jupiler League,KV Kortrijk,26,1.115,1.038,1.269,-0.231,0.308,0.192,0.500
4,2010/2011,Belgium Jupiler League,RSC Anderlecht,27,2.148,1.741,0.667,1.074,0.630,0.259,0.111
5,2010/2011,Belgium Jupiler League,KSV Cercle Brugge,25,1.040,0.960,1.320,-0.360,0.280,0.200,0.520
6,2010/2011,Belgium Jupiler League,Standard de Liège,26,1.538,1.423,1.308,0.115,0.462,0.154,0.385
7,2010/2011,Belgium Jupiler League,Sporting Charleroi,25,0.640,0.720,1.920,-1.200,0.120,0.280,0.600
8,2010/2011,Belgium Jupiler League,KRC Genk,26,2.077,1.923,0.885,1.038,0.615,0.231,0.154
9,2010/2011,Belgium Jupiler League,Lierse SK,3,1.000,0.667,0.667,0.000,0.000,1.000,0.000


## 5. Assess Team-Season Coverage

A partial season can produce unstable averages. Coverage is measured relative to the most complete team in each league-season, while league-seasons with fewer than 20 observed matches for even their most complete team are treated as too incomplete for reliable profiling.

In [5]:
team_season["league_season_max_matches"] = (
    team_season.groupby(["league_id", "season"])["matches_played"].transform("max")
)
team_season["coverage_ratio"] = (
    team_season["matches_played"] / team_season["league_season_max_matches"]
)

league_season_coverage = (
    team_season
    .groupby(["season", "league_id", "league_name"], as_index=False)
    .agg(
        teams=("team_api_id", "nunique"),
        min_matches=("matches_played", "min"),
        median_matches=("matches_played", "median"),
        max_matches=("matches_played", "max"),
    )
    .sort_values(["season", "league_name"])
)
incomplete_league_seasons = league_season_coverage.query("max_matches < 20")

display(team_season["coverage_ratio"].describe().to_frame().round(3))
display(incomplete_league_seasons)
print("Team-seasons below 80% coverage:", int((team_season["coverage_ratio"] < 0.80).sum()))

,coverage_ratio
count,1076.000
mean,0.958
std,0.119
min,0.062
25%,0.971
50%,1.000
75%,1.000
max,1.000


,season,league_id,league_name,teams,min_matches,median_matches,max_matches
6,2010/2011,15722,Poland Ekstraklasa,12,1,3.0,9
17,2011/2012,15722,Poland Ekstraklasa,13,1,11.0,16
33,2013/2014,1,Belgium Jupiler League,3,4,4.0,4


Team-seasons below 80% coverage: 36


### Decision

Incomplete league-seasons are removed rather than treated as full campaigns because short coverage can distort seasonal team profiles. The final sample keeps league-seasons with at least 20 available matches and team-seasons covering at least 80% of that league-season maximum.

## 6. Filter Incomplete Team-Seasons

In [6]:
valid_league_seasons = league_season_coverage.loc[
    league_season_coverage["max_matches"] >= 20,
    ["season", "league_id"],
]
team_season_clean = (
    team_season
    .merge(valid_league_seasons, on=["season", "league_id"], how="inner")
    .loc[lambda data: data["coverage_ratio"] >= 0.80]
    .copy()
)

filtering_summary = pd.Series({
    "before_filtering": len(team_season),
    "after_filtering": len(team_season_clean),
    "removed_team_seasons": len(team_season) - len(team_season_clean),
    "unique_teams": team_season_clean["team_api_id"].nunique(),
    "league_seasons": team_season_clean[["season", "league_id"]].drop_duplicates().shape[0],
}, name="value")
display(filtering_summary.to_frame())
display(team_season_clean[[
    "season", "league_name", "team_name", "matches_played",
    "league_season_max_matches", "coverage_ratio",
]].sort_values("coverage_ratio").head(15))

,value
before_filtering,1076
after_filtering,1028
removed_team_seasons,48
unique_teams,258
league_seasons,63


,season,league_name,team_name,matches_played,league_season_max_matches,coverage_ratio
879,2015/2016,Belgium Jupiler League,Royal Excel Mouscron,24,30,0.800000
455,2012/2013,Poland Ekstraklasa,Śląsk Wrocław,21,26,0.807692
1039,2015/2016,Switzerland Super League,FC Vaduz,26,32,0.812500
243,2011/2012,Italy Serie A,Novara,31,38,0.815789
645,2013/2014,Scotland Premier League,Ross County FC,29,35,0.828571
595,2013/2014,Netherlands Eredivisie,PEC Zwolle,25,30,0.833333
777,2014/2015,Netherlands Eredivisie,SC Cambuur,27,32,0.843750
462,2012/2013,Poland Ekstraklasa,Wisła Kraków,22,26,0.846154
451,2012/2013,Poland Ekstraklasa,P. Warszawa,22,26,0.846154
448,2012/2013,Poland Ekstraklasa,Ruch Chorzów,22,26,0.846154


### Finding

After applying the coverage rules, 1,028 team-season observations remain. The retained sample covers 258 teams and 63 league-seasons and excludes records that would be based on unusually short campaigns.

## 7. Validate and Save the Final Dataset

Only the filtered dataset is written to disk. This prevents the earlier incomplete version from being mistaken for the clean hand-off dataset.

In [7]:
duplicate_count = team_season_clean.duplicated(
    subset=["season", "league_id", "team_api_id"]
).sum()
missing_count = int(team_season_clean.isna().sum().sum())

final_checks = pd.Series({
    "team_season_observations": len(team_season_clean),
    "unique_teams": team_season_clean["team_api_id"].nunique(),
    "league_seasons": team_season_clean[["season", "league_id"]].drop_duplicates().shape[0],
    "missing_values": missing_count,
    "duplicate_team_seasons": duplicate_count,
}, name="value")
display(final_checks.to_frame())

assert len(team_season_clean) == 1_028
assert final_checks["unique_teams"] == 258
assert final_checks["league_seasons"] == 63
assert missing_count == 0
assert duplicate_count == 0

team_season_clean.to_csv(TEAM_DATA_PATH, index=False)
print(f"Saved clean team-season dataset to: {TEAM_DATA_PATH}")

,value
team_season_observations,1028
unique_teams,258
league_seasons,63
missing_values,0
duplicate_team_seasons,0


Saved clean team-season dataset to: data/processed/team_season_features.csv


## 8. Key Findings and Hand-Off

The source matches were reshaped into 35,218 team-match observations and aggregated into reliable team-season records. Coverage filtering produced a complete, duplicate-free dataset of 1,028 observations from 258 teams across 63 league-seasons.

`team_season_features.csv` is the sole output from this stage and is the input to the team-performance EDA notebook.